### Mutations modeled by Rosetta

* Look over the CHTC results that are stored in the sqlite DB (just scores)
* Create a test_train dataset
* Save the test set with redone rosetta terms 

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

import rosetta_db

## Mutations from 2D parent

In [2]:
df = rosetta_db.get_energy_scores(parent="2D")

In [3]:
df.columns

Index(['r_total_score', 'r_dslf_fa13', 'r_fa_atr', 'r_fa_dun', 'r_fa_elec',
       'r_fa_intra_rep', 'r_fa_intra_sol_xover4', 'r_fa_rep', 'r_fa_sol',
       'r_hbond_bb_sc', 'r_hbond_lr_bb', 'r_hbond_sc', 'r_hbond_sr_bb',
       'r_lk_ball_wtd', 'r_omega', 'r_p_aa_pp', 'r_pro_close', 'r_rama_prepro',
       'r_ref', 'r_yhh_planarity', 'd_total_score', 'd_Grid_score',
       'd_Transform_accept_ratio', 'd_angle_constraint',
       'd_atom_pair_constraint', 'd_chainbreak', 'd_classic_grid_X',
       'd_coordinate_constraint', 'd_dihedral_constraint', 'd_dslf_ca_dih',
       'd_dslf_cs_ang', 'd_dslf_ss_dih', 'd_dslf_ss_dst', 'd_fa_atr',
       'd_fa_dun', 'd_fa_elec', 'd_fa_pair', 'd_fa_rep', 'd_fa_sol',
       'd_hbond_bb_sc', 'd_hbond_lr_bb', 'd_hbond_sc', 'd_hbond_sr_bb',
       'd_if_X_angle_constraint', 'd_if_X_atom_pair_constraint',
       'd_if_X_chainbreak', 'd_if_X_coordinate_constraint',
       'd_if_X_dihedral_constraint', 'd_if_X_dslf_ca_dih',
       'd_if_X_dslf_cs_ang', 'd_i

In [4]:
df.head()

,r_total_score,r_dslf_fa13,r_fa_atr,r_fa_dun,r_fa_elec,r_fa_intra_rep,r_fa_intra_sol_xover4,r_fa_rep,r_fa_sol,r_hbond_bb_sc,...,d_if_X_pro_close,d_if_X_ref,d_interface_delta_X,d_ligand_is_touching_X,d_omega,d_p_aa_pp,d_pro_close,d_ref,d_total_score_X,variant
0,-857.943,0.0,-1563.589,316.642,-539.426,2.838,55.569,182.141,936.300,-57.349,...,0.0,0.0,-10.014,1.0,47.292,-40.692,1.027,-41.04,-30.0,D53W.V166Y.W196K
1,-853.451,0.0,-1566.619,320.550,-537.542,2.932,56.286,186.453,929.978,-55.907,...,0.0,0.0,-7.627,1.0,49.498,-41.948,1.125,-43.43,-30.0,A16K.N167I.A187M
2,-847.306,0.0,-1562.663,321.941,-540.116,2.886,56.374,181.451,931.672,-55.895,...,0.0,0.0,-8.558,1.0,48.730,-39.806,0.964,-39.23,-31.0,V43C.A73I.L137W
3,-862.828,0.0,-1561.831,318.629,-539.675,2.875,56.133,180.080,939.619,-57.035,...,0.0,0.0,-8.589,1.0,50.224,-41.688,1.038,-44.25,-28.0,V39E.Y213E.A247R
4,-846.302,0.0,-1558.842,320.008,-537.013,2.821,56.273,185.282,939.990,-57.288,...,0.0,0.0,-7.669,1.0,49.855,-42.352,1.008,-41.38,-29.0,K88L.L136E.A187H


In [5]:
# Number of mutations modeled
len(df)

100000

In [6]:
# campaign1_muts = [
#     'D157G.I71V.R172H.F152L.I38V.Q233R.F261L',
#     'D157G.I71V.R172H.F152L.I38V.Q233R.F261L.V38I.R48C',
#     'D157G.I71V.R172H.F152L',
#     'D157G.I71V.R172H.N65D',
#     'D157G.I71V.R172H',
#     'D157G.I71V',
#     'D157G'
# ]

## Create test_train split

In [7]:
# Set to True if we really want to replace
# Setting to False so that we do not accidentally overwrite
REPLACE_TEST_TRAIN_SPLIT = True
splits_dir = rosetta_db.get_splits_dir(parent="2D")

if REPLACE_TEST_TRAIN_SPLIT:
    np.random.seed(42)# set seed for test_training split
    train_df, test_df = train_test_split(df, test_size=0.1)
    
    train_df.reset_index(drop=True).to_pickle(splits_dir / "train_df.pkl")
    test_df.reset_index(drop=True).to_pickle(splits_dir / "test_df.pkl")
     
    train_df.variant.sort_values().to_csv(splits_dir / "train_variants.txt", 
                            header=False, index=False)
    test_df.variant.sort_values().to_csv(splits_dir / "test_variants.txt", 
                            header=False, index=False)


    

In [8]:
# Check the variants with the saved backup list of variants
# to make sure that nothing has changed
!diff {splits_dir}/train_variants.txt \
    {splits_dir}/backup/train_variants.sort.txt
!diff {splits_dir}/test_variants.txt\
    {splits_dir}/backup/test_variants.sort.txt

In [9]:
# the test set was redone a second time to account for
# stochasticty in the rosetta results
if REPLACE_TEST_TRAIN_SPLIT:

    test2_df = rosetta_db.get_energy_scores(parent="2D", 
                   table_name_postfix="_redo_test_set")
    test2_df.reset_index(drop=True).to_pickle(splits_dir / "test2_df.pkl")
    
    test2_df.variant.sort_values().to_csv(splits_dir / "test2_variants.txt", 
                            header=False, index=False)

In [10]:
!diff {splits_dir}/test2_variants.txt\
    {splits_dir}/backup/test_variants.sort.txt

### Looking at sum of energy terms

In [11]:
# drop the first r_energy term (r_total_score) and sum up the other r_ energy terms
# we should get something very close to the r_total_score
print(f"Sum of r_ energy terms for first variant : {df.loc[0, df.columns.str.startswith('r_')][1:].sum():.4f}")
print(f"    Total energy terms for first variant : {df.loc[0, 'r_total_score']:.4f}")

Sum of r_ energy terms for first variant : -857.9450
    Total energy terms for first variant : -857.9430


In [12]:
# The same thing isn't true for the d_ scores
# drop the first d_energy term (d_total_score) and sum up the other d_ energy terms
# we do not get something very close to the d_total_score
print(f"Sum of d_ energy terms for first variant : {df.loc[0, df.columns.str.startswith('d_')][1:].sum():.4f}")
print(f"    Total energy terms for first variant : {df.loc[0, 'd_total_score']:.4f}")

Sum of d_ energy terms for first variant : -1010.2990
    Total energy terms for first variant : -900.9790


In [13]:
with pd.option_context("display.max_rows", 1000):
    display(df.describe().transpose())


,count,mean,std,min,25%,50%,75%,max
r_total_score,100000.0,-855.129265,20.501256,-887.225,-861.69700,-856.9500,-851.28575,28.101
r_dslf_fa13,100000.0,0.000000,0.000000,0.000,0.00000,0.0000,0.00000,0.000
r_fa_atr,100000.0,-1560.950925,7.664767,-1654.659,-1565.67600,-1561.0220,-1556.20975,-1520.674
r_fa_dun,100000.0,315.848786,3.634320,301.594,313.35900,315.6800,318.13700,340.720
r_fa_elec,100000.0,-537.106482,5.717072,-566.892,-540.28400,-536.9220,-533.53400,-503.496
r_fa_intra_rep,100000.0,2.851107,0.047701,2.708,2.82100,2.8470,2.87500,4.199
r_fa_intra_sol_xover4,100000.0,55.399383,0.717147,51.867,54.96700,55.4370,55.86200,61.722
r_fa_rep,100000.0,184.715443,18.397274,174.592,182.27500,183.6870,185.35900,1066.617
r_fa_sol,100000.0,929.772388,7.763332,890.560,924.63700,929.5630,934.70800,966.306
r_hbond_bb_sc,100000.0,-55.903234,1.413368,-63.683,-56.67300,-55.9920,-55.15900,-47.971
